# UCA Direction Finding Using I/Q Phase Correlation

This notebook develops the direction-finding simulation from the received I/Q samples.

The relative phases between neighbouring antenna signals are extracted from the received I/Q data and compared with pre-calculated phase signatures to estimate the transmitter azimuth.

In [ ]:
using FFTW
using LinearAlgebra
using Plots
plotly() # plotyly backend for Plots

In [ ]:
j = im

# Speed of light
c = 3e8

# Receiver sampling frequency
fs = 20e6
dt = 1 / fs

# Number of samples
Nsamples = 2^15

# Duration of one antenna acquisition
Tcapture = Nsamples / fs

# Time vector
t = (0:Nsamples - 1) * dt
t_max = Nsamples * dt

# Reciver centre frequency and transmitted frequency
fc = 433.0e6      # Receiver centre frequency = 433.0 MHz
f0 = fc + 100e3

# Wavelength of the transmitted signal
λ = c / f0

@show Tcapture

### Sequential Antenna Acquisition

Because the antennas are sampled at different times, the measured phase difference contains an additional switching-time phase:

$$
\Delta\phi_{nm,\mathrm{measured}}
=
\phi_n-\phi_m
+
2\pi f_{\mathrm{BB}}(t_n-t_m).
$$

In [ ]:
# Number of antennas on the circle
Nant = 6

# Acquisition start time for each antenna
t_antenna = zeros(Nant)

for n = 1:Nant
    t_antenna[n] = (n - 1) * Tcapture
end

# Circle radius
Rcircle = 0.4 * λ

# Store the antenna positions
A = [zeros(3) for n in 1:Nant]

for n = 1:Nant
    
    theta = (2π / Nant) * (n - 1)
    
    x = Rcircle * cos(theta)
    y = Rcircle * sin(theta)
    z = 0
    
    A[n] = [x, y, z]
    
end

In [ ]:
# True source azimuth and distance used only to generate the simulation
phi0 = deg2rad(30)
Rsource = 10

# Reflected-path direction
theta_reflection = 100
phi_reflection = deg2rad(theta_reflection)

# Source position
P = [Rsource * cos(phi0), Rsource * sin(phi0), 0]

# Virtual reflected source position
P_reflection = [Rsource * cos(phi_reflection), Rsource * sin(phi_reflection), 0]

# Store distance and propagation delay for each antenna
range = zeros(Nant)
tau = zeros(Nant)

# Store reflected-path distance and delay
range_reflection = zeros(Nant)
tau_reflection = zeros(Nant)

for n = 1:Nant
    
    # Direct path
    R = A[n] - P
    
    range[n] = norm(R)
    tau[n] = range[n] / c
    
    # Reflected path
    R_reflection = A[n] - P_reflection
    
    range_reflection[n] = norm(R_reflection)
    tau_reflection[n] = range_reflection[n] / c
end

@show range
@show tau
@show range_reflection
@show tau_reflection

## 3. I/Q Data at Each Antenna

The signal transmitted at RF frequency $f_0$ is delayed by the propagation time to each antenna.

For antenna $n$,

$$
\tau_n = \frac{r_n}{c}
$$

and the received RF signal is

$$
x_n(t)
=
A e^{j2\pi f_0(t-\tau_n)}.
$$

Expanding the delay term,

$$
x_n(t)
=
A e^{-j2\pi f_0\tau_n}
e^{j2\pi f_0t}.
$$

The receiver is centred at frequency $f_c$. After downconversion, the
I/Q signal becomes

$$
v_n(t)
=
A e^{-j2\pi f_0\tau_n}
e^{j2\pi(f_0-f_c)t}.
$$

The term

$$
e^{-j2\pi f_0\tau_n}
$$

contains the propagation phase caused by the distance from the source
to antenna $n$.

The baseband frequency is

$$
f_{\mathrm{BB}} = f_0-f_c.
$$

### Additive Gaussian Noise

Receiver noise is modelled as additive complex Gaussian noise

$$
n[k] = n_I[k] + jn_Q[k],
$$

where

$$
n_I,n_Q \sim \mathcal{N}(0,\sigma^2).
$$

For the complex I/Q signal, the signal-to-noise ratio is

$$
\mathrm{SNR}
=
\frac{P_s}{P_n}.
$$

Since the signal magnitude is $A$,

$$
P_s=A^2,
$$

while the complex noise power is

$$
P_n=2\sigma^2.
$$

Therefore,

$$
\boxed{
\sigma
=
\sqrt{
\frac{A^2}
{(2)*\,10^{\mathrm{SNR}_{dB}/10}}
}
}
$$

The noise is added directly to the received I/Q samples before FFT processing.

In [ ]:
# Signal amplitude
Amp = 1

# Reflected-path relative amplitude
α = 0.3

# Baseband frequency
fBB = f0 - fc

# Signal-to-noise ratio
SNR_dB = 10

# Noise standard deviation
σ = sqrt(Amp^2 / (2 * 10^(SNR_dB / 10)))
#σ = 0

# Store the received I/Q samples for each antenna
IQdata = [zeros(ComplexF64, Nsamples) for n in 1:Nant]

for n = 1:Nant
    
    # Phase shift caused by propagation to antenna n
    phase_delay_direct = exp(-j * 2π * f0 * tau[n])
    
    # Received signal
    direct_signal = Amp * phase_delay_direct .* exp.(j * 2π * fBB .* (t .+ t_antenna[n]))
    
    # Reflected path
    phase_delay_reflection = exp(-j * 2π * f0 * tau_reflection[n])
    
    reflected_signal = α * Amp * phase_delay_reflection .* exp.(j * 2π * fBB .* (t .+ t_antenna[n]))
    
    signal = direct_signal + reflected_signal
    
    # Complex Gaussian noise
    noise = σ .* randn(Nsamples) .+ j .* σ .* randn(Nsamples)
    
    # I/Q signal received at antenna n
    IQdata[n] = signal + noise
end
@show SNR_dB
@show σ

In [ ]:
# Complex Gaussian noise visualisation

noise_plot = σ .* randn(Nsamples) .+ j .* σ .* randn(Nsamples)
histogram(
    real.(noise_plot), 
    bin = 60, 
    xlabel = "Noise Amplitude", 
    ylabel = "Number of Samples", 
    title = "Gaussian Noise - I Cmponent",
    label = false
)

In [ ]:
# Store the FFT for each antenna
FFTdata = [zeros(ComplexF64, Nsamples) for n in 1:Nant]

for n = 1:Nant
    FFTdata[n] = fft(IQdata[n])
end


# Use antenna 1 to find the signal frequency
magnitude = abs.(FFTdata[1])

peak_magnitude, peak_index = findmax(magnitude)

df = fs / Nsamples
peak_frequency = (peak_index - 1) * df

@show peak_index
@show peak_frequency

In [ ]:
# Frequency axis
frequency_axis = (0:Nsamples-1) .* df

# Plot FFT magnitude of antenna 1
p = plot(
    frequency_axis,
    abs.(FFTdata[1]),
    xlabel = "Frequency (Hz)",
    ylabel = "FFT Magnitude",
    title = "FFT Spectrum of All Antennas",
    label = "Antenna 1",
    xlims = (0, 300e3)
)

# Add antennas 2 to 6
for n = 2:Nant
    plot!(
        p,
        frequency_axis,
        abs.(FFTdata[n]),
        label = "Antenna $n"
    )
end

display(p)

In [ ]:
# Complex FFT peak value for each antenna
FFT_peak = zeros(ComplexF64, Nant)

for n = 1:Nant
    FFT_peak[n] = FFTdata[n][peak_index]
end


# Plot the complex FFT peak values
p = scatter(
    real.(FFT_peak),
    imag.(FFT_peak),
    xlabel = "Real (I)",
    ylabel = "Imaginary (Q)",
    title = "Complex FFT Peak Values",
    label = false,
    aspect_ratio = :equal
)

# Label each antenna
for n = 1:Nant
    annotate!(
        p,
        real(FFT_peak[n]),
        imag(FFT_peak[n]),
        text("A$n", 9)
    )
end

display(p)

In [ ]:
FFT_phase_deg = zeros(Nant)

for n = 1:Nant
    FFT_phase_deg[n] = angle(FFT_peak[n]) * 180 / π
end

@show FFT_phase_deg

## 4. Relative Phase Between Antennas

The received baseband I/Q signal at antenna $n$ is

$$
v_n(t)
=
A e^{-j2\pi f_0\tau_n}
e^{j2\pi f_{BB}t},
$$

where

$$
f_{BB}=f_0-f_c.
$$

The propagation phase at antenna $n$ is

$$
\phi_n
=
-2\pi f_0\tau_n
=
-\frac{2\pi r_n}{\lambda}.
$$

The sampled signal is therefore

$$
v_n[k]
=
A e^{j\phi_n}
e^{j2\pi f_{BB}k/f_s}.
$$

The FFT of the received I/Q samples is

$$
V_n[m]
=
\sum_{k=0}^{N-1}
v_n[k]e^{-j2\pi mk/N}.
$$

The signal is identified from the FFT peak

$$
m_{\text{peak}}
=
\underset{m}{\operatorname{arg\,max}}
|V_n[m]|.
$$

The complex FFT value at this bin is

$$
V_n
=
V_n[m_{\text{peak}}]
=
B_n e^{j\psi_n}.
$$

For neighbouring antennas $n$ and $m$,

$$
V_nV_m^*
=
B_nB_m e^{j(\psi_n-\psi_m)},
$$

therefore the relative phase is

$$
\boxed{
\Delta\phi_{nm}
=
\angle(V_nV_m^*)
}
$$

The six-element circular array therefore produces the measured phase
signature

$$
\boxed{
\mathbf{p}_{\text{measured}}
=
\begin{bmatrix}
\Delta\phi_{12} &
\Delta\phi_{23} &
\Delta\phi_{34} &
\Delta\phi_{45} &
\Delta\phi_{56} &
\Delta\phi_{61}
\end{bmatrix}.
}
$$

This phase signature is compared with the pre-calculated signatures to
estimate the source azimuth.

In [ ]:
# Switching phase contribution between neighbouring antennas
switch_phase_deg = zeros(Nant)

for n = 1:Nant
    
    # Select neighbouring antenna
    if n < Nant
        next_n = n + 1
    else
        next_n = 1
    end
    
    # Switching-time phase contribution
    switch_phase_deg[n] = 2π * fBB * (t_antenna[n] - t_antenna[next_n]) * 180 / π
    
    # Wrap to -180 to 180 degrees
    switch_phase_deg[n] = angle(exp(j * switch_phase_deg[n] * π / 180)) * 180 / π
end
@show switch_phase_deg

In [ ]:
# Phase difference between neighbouring antennas
phase_difference_deg = zeros(Nant)

# Phase differnce after correcting for acquisition time
phase_difference_corrected_deg = zeros(Nant)

for n = 1:Nant

    # Select the next antenna
    if n < Nant
        next_n = n + 1
    else
        next_n = 1
    end

    # Complex FFT peak values
    V1 = FFT_peak[n]
    V2 = FFT_peak[next_n]

    # Relative phase
    phase_difference_deg[n] = angle(V1 * conj(V2)) * 180 / π
    
    # Phase caused by the different acquisition times
    switch_phase = 2π * fBB * (t_antenna[n] - t_antenna[next_n])
    
    # Remove the switching phase and wrap to -180 to 180 degrees
    phase_difference_corrected_deg[n] = angle(exp(j * (phase_difference_deg[n] * π / 180 - switch_phase))) * 180 / π

end

@show phase_difference_deg
@show phase_difference_corrected_deg

### Far-Field Phase Lookup

The received I/Q signals are generated using the exact source-to-antenna
distances

$$
r_n = \|\mathbf{A}_n-\mathbf{P}\|.
$$

The phase lookup table assumes that the transmitter is sufficiently far from
the array for the incoming wavefront to be approximated as planar. The
path-length difference between neighbouring antennas is then

$$
r_n-r_m
\approx
-\hat{\mathbf{u}}\cdot(\mathbf{A}_n-\mathbf{A}_m),
$$

giving the expected phase difference

$$
\Delta\phi_{nm}
\approx
\frac{2\pi}{\lambda}
\hat{\mathbf{u}}\cdot(\mathbf{A}_n-\mathbf{A}_m).
$$

In [ ]:
# Azimuth angles to pre-calculate
azimuth_scan_deg = 0:359

# Store the 6 phase differences for every azimuth
phase_table = zeros(length(azimuth_scan_deg), Nant)

for angle_index = 1:length(azimuth_scan_deg)
    
    azimuth = azimuth_scan_deg[angle_index] * π / 180
    
    # Unit vector pointing towards this azimuth
    u = [cos(azimuth), sin(azimuth), 0]
    
    for n = 1:Nant
        
        # Select neighbouring antenna
        if n < Nant
            next_n = n + 1
        else
            next_n = 1
        end
        
        # Distance difference between the two antenna positions
        delta_r = dot(u, A[n] - A[next_n])
        
        # Corresponding phase difference
        delta_phase = (2π / λ) * delta_r
        
        # Store phase wrapped between -180 and 180 degrees
        phase_table[angle_index, n] = angle(exp(j * delta_phase)) * 180 / π
        
    end
end

In [ ]:
# Creating a table with the RMSE values that result from comparing the actual phase difference and the one
# stored in a lookup table

RMSE_table = zeros(length(azimuth_scan_deg))

for angle_index = 1:length(azimuth_scan_deg)

    difference = phase_difference_corrected_deg - phase_table[angle_index, :]

    wrapped_difference = angle.(exp.(j .* difference .* π / 180)) .* 180 / π

    squared_difference = wrapped_difference .^ 2

    RMSE_table[angle_index] = sqrt(sum(squared_difference) / Nant)

end

minimum_RMSE, minimum_index = findmin(RMSE_table)

estimated_azimuth = azimuth_scan_deg[minimum_index]

@show minimum_index
@show minimum_RMSE
@show estimated_azimuth

In [ ]:
plot(
    azimuth_scan_deg,
    RMSE_table,
    xlabel = "Azimuth (degrees)",
    ylabel = "RMSE (degrees)",
    title = "RMSE versus Candidate Azimuth",
    legend = false
)

## Simulation Test Summary

### Baseline

| Variable changed | Value | Estimated azimuth | Minimum RMSE |
|---|---:|---:|---:|
| — | Basic estimator | $30^\circ$ | $1.221^\circ$ |

### Source Range Test

The source distance **`Rsource`** was varied while the true azimuth remained
$\theta_0 = 30^\circ$.

| **`Rsource`** (m) | Estimated azimuth | Minimum RMSE |
|---:|---:|---:|
| 0.50 | $30^\circ$ | $20.89^\circ$ |
| 0.75 | $30^\circ$ | $14.85^\circ$ |
| 1.00 | $30^\circ$ | $11.50^\circ$ |
| 2.00 | $30^\circ$ | $6.22^\circ$ |
| 5.00 | $30^\circ$ | $2.47^\circ$ |
| 10.00 | $30^\circ$ | $1.25^\circ$ |

### Sequential Acquisition Test

Sequential antenna acquisition was introduced using **`t_antenna`**.

| Condition | Estimated azimuth | Minimum RMSE |
|---|---:|---:|
| Switching phase not corrected | $29^\circ$ | $60.17^\circ$ |
| Switching phase corrected | $30^\circ$ | $1.221^\circ$ |

### Noise Test

Gaussian noise was enabled with **`SNR_dB = 20`**, while multipath was disabled
and the switching-phase correction remained active.

| **`SNR_dB`** | Estimated azimuth | Minimum RMSE |
|---:|---:|---:|
| 20 dB | $30^\circ$ | $1.216^\circ$ |

### Multipath Test

The reflected-path relative amplitude **`α`** was varied with the reflected
path arriving from $100^\circ$. The SNR was $20$ dB and switching-phase
correction remained active.

| **`α`** | Estimated azimuth | Bearing error | Minimum RMSE |
|---:|---:|---:|---:|
| 0.1 | $31^\circ$ | $1^\circ$ | $2.884^\circ$ |
| 0.2 | $33^\circ$ | $3^\circ$ | $5.072^\circ$ |
| 0.3 | $35^\circ$ | $5^\circ$ | $7.444^\circ$ |
| 0.5 | $41^\circ$ | $11^\circ$ | $11.936^\circ$ |
| 0.8 | $55^\circ$ | $25^\circ$ | $16.899^\circ$ |
| 1.0 | $65^\circ$ | $35^\circ$ | $18.428^\circ$ |